In [ ]:
# ============================================
# OS1 — ANALYSE EXPLORATOIRE DAiSEE
# Cellule 1 : Imports et chargement
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Charger les 3 datasets
train = pd.read_csv('../data/DAiSEE/Labels/TrainLabels.csv')
test = pd.read_csv('../data/DAiSEE/Labels/TestLabels.csv')
val = pd.read_csv('../data/DAiSEE/Labels/ValidationLabels.csv')

# Corriger les noms de colonnes
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()
val.columns = val.columns.str.strip()

# Définir les états
etats = ['Boredom', 'Engagement', 'Confusion', 'Frustration']

# Informations générales
print("=== INFORMATIONS GÉNÉRALES ===")
print(f"Train      : {len(train)} clips")
print(f"Validation : {len(val)} clips")
print(f"Test       : {len(test)} clips")
print(f"Total      : {len(train)+len(val)+len(test)} clips")
print("\n=== APERÇU TRAIN ===")
print(train.head())
print("\n=== STATISTIQUES ===")
print(train.describe()) # moyennes, écarts-types, min/max par dimension.



In [ ]:
# ============================================
# Cellule 2 : Distribution des 4 états
# ============================================

couleurs = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribution des états émotionnels — DAiSEE Train',
             fontsize=16, fontweight='bold')

for i, (etat, couleur) in enumerate(zip(etats, couleurs)):
    ax = axes[i//2, i%2]
    counts = train[etat].value_counts().sort_index()
    bars = ax.bar(counts.index, counts.values, color=couleur, edgecolor='white')
    ax.set_title(f'{etat}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Niveau (0=très faible, 3=très élevé)')
    ax.set_ylabel('Nombre de clips')
    ax.set_xticks([0, 1, 2, 3])
    for bar, count in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(count), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../data/distribution_etats.png', dpi=150)
plt.show()
print("Distribution sauvegardée !")

In [ ]:
# ============================================
# Cellule 3 : Matrice de corrélation
# ============================================

fig, ax = plt.subplots(figsize=(8, 6))

corr = train[etats].corr()
# Montre les liens entre dimensions (ex. Engagement ↔ Boredom négatif).

sns.heatmap(corr,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            ax=ax,
            linewidths=0.5,
            square=True)

ax.set_title('Corrélation entre états émotionnels — DAiSEE',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/correlation_etats.png', dpi=150)
plt.show()
print("Matrice de corrélation sauvegardée !")

In [ ]:
# ============================================
# Cellule 4 : Boxplots  Compare la dispersion des 4 états sur le train.
# ============================================

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('Dispersion des états émotionnels — DAiSEE',
             fontsize=14, fontweight='bold')

# Boxplot — Les 4 états
ax.boxplot(
    [train['Boredom'], train['Engagement'],
     train['Confusion'], train['Frustration']],
    tick_labels=['Boredom', 'Engagement', 'Confusion', 'Frustration'],
    patch_artist=True,
    boxprops=dict(facecolor='lightblue'),
    medianprops=dict(color='red', linewidth=2)
)
ax.set_title('Dispersion des 4 états', fontsize=12)
ax.set_ylabel('Niveau (0-3)')
ax.set_xlabel('État émotionnel')

plt.tight_layout()
plt.savefig('../data/boxplots_etats.png', dpi=150)
plt.show()
print("Boxplots sauvegardés !")

In [ ]:
# ============================================
# Cellule 5 : Patterns critiques
 #Logique (filtres sur le train) :
# ============================================

print("=== PATTERNS CRITIQUES ===\n")

p1 = train[(train['Confusion'] >= 2) & (train['Engagement'] <= 1)]
print(f"Pattern 1 — Confusion haute + Engagement faible : {len(p1)} clips ({len(p1)/len(train)*100:.1f}%)")

p2 = train[train['Frustration'] >= 2]
print(f"Pattern 2 — Frustration élevée : {len(p2)} clips ({len(p2)/len(train)*100:.1f}%)")
# Pattern critique — Ennui élevé
p3 = train[train['Boredom'] >= 2]
print(f"Pattern 3 — Ennui élevé : {len(p3)} clips ({len(p3)/len(train)*100:.1f}%)")

p4 = train[train['Engagement'] == 0]
print(f"Pattern 4 — Engagement nul : {len(p4)} clips ({len(p4)/len(train)*100:.1f}%)")

p5 = train[(train['Confusion'] >= 2) &
           (train['Frustration'] >= 2) &
           (train['Engagement'] <= 1)]
print(f"Pattern 5 — Situation critique totale : {len(p5)} clips ({len(p5)/len(train)*100:.1f}%)")

p6 = train[train['Engagement'] >= 2]
print(f"Pattern 6 — Bon engagement : {len(p6)} clips ({len(p6)/len(train)*100:.1f}%)")

print(f"\nTotal clips analysés : {len(train)}")


In [ ]:
# ============================================
# Cellule 6 : Transitions critiques
# Trie les clips par ClipID (ordre temporel implicite).

# ============================================

print("=== TRANSITIONS CRITIQUES ===\n")

train_sorted = train.sort_values('ClipID').reset_index(drop=True)
train_sorted.columns = train_sorted.columns.str.strip()

# Transition Engagement → Boredom
transitions_eb = []
for i in range(1, len(train_sorted)):
    prev = train_sorted.iloc[i-1]
    curr = train_sorted.iloc[i]
        # Détecte quand engagement baisse ET ennui monte

    if prev['Engagement'] >= 2 and curr['Engagement'] <= 1 and curr['Boredom'] >= 2:
        transitions_eb.append({
            'clip_avant': prev['ClipID'],
            'clip_apres': curr['ClipID'],
            'engagement_avant': prev['Engagement'],
            'engagement_apres': curr['Engagement'],
            'boredom_apres': curr['Boredom']
        })

# Transition Engagement → Confusion
transitions_ec = []
for i in range(1, len(train_sorted)):
    prev = train_sorted.iloc[i-1]
    curr = train_sorted.iloc[i]
    if prev['Engagement'] >= 2 and curr['Engagement'] <= 1 and curr['Confusion'] >= 2:
        transitions_ec.append({
            'clip_avant': prev['ClipID'],
            'clip_apres': curr['ClipID'],
            'engagement_avant': prev['Engagement'],
            'engagement_apres': curr['Engagement'],
            'confusion_apres': curr['Confusion']
        })

print(f"Transitions Engagement → Boredom  : {len(transitions_eb)} cas")
print(f"Transitions Engagement → Confusion : {len(transitions_ec)} cas")
print(f"Total transitions critiques        : {len(transitions_eb)+len(transitions_ec)} cas")

print("\n--- Exemples Engagement → Boredom ---")
for t in transitions_eb[:3]:
    print(f"  {t['clip_avant']} → {t['clip_apres']} | Engagement {t['engagement_avant']}→{t['engagement_apres']} | Boredom={t['boredom_apres']}")

print("\n--- Exemples Engagement → Confusion ---")
for t in transitions_ec[:3]:
    print(f"  {t['clip_avant']} → {t['clip_apres']} | Engagement {t['engagement_avant']}→{t['engagement_apres']} | Confusion={t['confusion_apres']}")

In [ ]:
# ============================================
# Cellule 7 : Seuils d'alerte
# ============================================

print("=== SEUILS D'ALERTE ===\n")

# Boredom ≥ 2 pendant 3 clips consécutifs
alertes_boredom = []
compteur = 0
for i in range(len(train_sorted)):
    if train_sorted.iloc[i]['Boredom'] >= 2:
        compteur += 1
        if compteur >= 3:
            alertes_boredom.append(train_sorted.iloc[i]['ClipID'])
    else:
        compteur = 0

# Confusion ≥ 2 pendant 2 clips consécutifs
alertes_confusion = []
compteur = 0
for i in range(len(train_sorted)):
    if train_sorted.iloc[i]['Confusion'] >= 2:
        compteur += 1
        if compteur >= 2:
            alertes_confusion.append(train_sorted.iloc[i]['ClipID'])
    else:
        compteur = 0

# Frustration ≥ 2 pendant 2 clips consécutifs
alertes_frustration = []
compteur = 0
for i in range(len(train_sorted)):
    if train_sorted.iloc[i]['Frustration'] >= 2:
        compteur += 1
        if compteur >= 2:
            alertes_frustration.append(train_sorted.iloc[i]['ClipID'])
    else:
        compteur = 0

print(f"Alertes Boredom ≥ 2 (3 clips)    : {len(alertes_boredom)} cas")
print(f"Alertes Confusion ≥ 2 (2 clips)  : {len(alertes_confusion)} cas")
print(f"Alertes Frustration ≥ 2 (2 clips): {len(alertes_frustration)} cas")
print(f"Total alertes                     : {len(alertes_boredom)+len(alertes_confusion)+len(alertes_frustration)} cas")

In [ ]:
# ============================================
# Cellule 8 : Segmentation 70/15/15
# Protocole méthodologique pour évaluation future (pas le flux UI).

print("=== SEGMENTATION 70/15/15 ===\n")

all_data = pd.concat([train, val, test], ignore_index=True)
all_data.columns = all_data.columns.str.strip()
all_data = all_data.sample(frac=1, random_state=42).reset_index(drop=True)

total = len(all_data)
taille_conception = int(total * 0.70)
taille_validation = int(total * 0.15)

conception = all_data[:taille_conception]
validation = all_data[taille_conception:taille_conception + taille_validation]
test_final = all_data[taille_conception + taille_validation:]

print(f"Ensemble conception (70%)  : {len(conception)} clips")
print(f"Ensemble validation (15%)  : {len(validation)} clips")
print(f"Ensemble test (15%)        : {len(test_final)} clips")
print(f"Total vérifié              : {len(conception)+len(validation)+len(test_final)} clips")

conception.to_csv('../data/DAiSEE/Labels/conception_70.csv', index=False)
validation.to_csv('../data/DAiSEE/Labels/validation_15.csv', index=False)
test_final.to_csv('../data/DAiSEE/Labels/test_15.csv', index=False)
print("\nSegmentation sauvegardée !")

In [ ]:
# ============================================
# Cellule 9 : 50 scénarios représentatifs
# ============================================

scenarios = []
# 15 scénarios stables

stable = train[train['Engagement'] >= 2].sample(15, random_state=42).copy()
stable['pattern'] = 'stable'
scenarios.append(stable)
# 10 scénarios ennui élevé

ennui = train[train['Boredom'] >= 2].sample(10, random_state=42).copy()
ennui['pattern'] = 'ennui_eleve'
scenarios.append(ennui)

frustration = train[train['Frustration'] >= 2].sample(8, random_state=42).copy()
frustration['pattern'] = 'frustration'
scenarios.append(frustration)

confusion = train[train['Confusion'] >= 2].sample(7, random_state=42).copy()
confusion['pattern'] = 'confusion'
scenarios.append(confusion)

eng_faible = train[train['Engagement'] <= 1].sample(6, random_state=42).copy()
eng_faible['pattern'] = 'engagement_faible'
scenarios.append(eng_faible)

critiques = train[(train['Confusion'] >= 2) & (train['Frustration'] >= 2)]
critiques = critiques.sample(min(4, len(critiques)), random_state=42).copy()
critiques['pattern'] = 'critique'
scenarios.append(critiques)
# Combiner et sauvegarder

df_scenarios = pd.concat(scenarios, ignore_index=True)

print(f"Total scénarios : {len(df_scenarios)}")
print(f"\nRépartition :")
print(df_scenarios['pattern'].value_counts())

df_scenarios.to_csv('../data/scenarios_50.csv', index=False)
print("\n50 scénarios sauvegardés !")

In [ ]:
# ============================================
# Cellule 10 : Trajectoires temporelles
#Courbes d’évolution des 4 scores sur des séquences de clips.
train_sorted = train.sort_values('ClipID').reset_index(drop=True)
train_sorted.columns = train_sorted.columns.str.strip()
train_sorted['StudentID'] = train_sorted['ClipID'].str[:4]

etudiants = train_sorted['StudentID'].unique()[:3]
couleurs = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Trajectoires temporelles des états émotionnels — DAiSEE',
             fontsize=14, fontweight='bold')

for idx, etudiant in enumerate(etudiants):
    ax = axes[idx]
    data_etudiant = train_sorted[train_sorted['StudentID'] == etudiant].reset_index(drop=True)

    for etat, couleur in zip(etats, couleurs):
        ax.plot(data_etudiant.index,
                data_etudiant[etat],
                label=etat,
                color=couleur,
                linewidth=2,
                marker='o',
                markersize=4)

    ax.set_title(f'Étudiant {etudiant} — Évolution des états', fontsize=11)
    ax.set_xlabel('Séquence de clips')
    ax.set_ylabel('Niveau (0-3)')
    ax.set_yticks([0, 1, 2, 3])
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

    for i in range(len(data_etudiant)):
        if data_etudiant.iloc[i]['Boredom'] >= 2:
            ax.axvspan(i-0.5, i+0.5, alpha=0.1, color='red')

plt.tight_layout()
plt.savefig('../data/trajectoires_temporelles.png', dpi=150)
plt.show()
print("Trajectoires temporelles sauvegardées !")

In [ ]:
# ============================================
# Cellule 11 : Résumé final OS1
# ============================================

print("=" * 50)
print("        RÉSUMÉ FINAL OS1 — DAiSEE")
print("=" * 50)
print(f"\n📊 Dataset analysé     : {len(train)+len(val)+len(test)} clips total")
print(f"📈 Patterns identifiés : 6 patterns")
print(f"🔄 Transitions         : {len(transitions_eb)+len(transitions_ec)} cas")
print(f"⚠️  Alertes confirmées  : {len(alertes_boredom)+len(alertes_confusion)+len(alertes_frustration)} cas")
print(f"🎯 Scénarios créés     : 50 scénarios")
print(f"\n📁 Fichiers sauvegardés :")
print(f"   - data/distribution_etats.png")
print(f"   - data/correlation_etats.png")
print(f"   - data/boxplots_etats.png")
print(f"   - data/trajectoires_temporelles.png")
print(f"   - data/scenarios_50.csv")
print(f"   - data/DAiSEE/Labels/conception_70.csv")
print(f"   - data/DAiSEE/Labels/validation_15.csv")
print(f"   - data/DAiSEE/Labels/test_15.csv")
print(f"\n✅ OS1 TERMINÉ AVEC SUCCÈS !")
print("=" * 50)